# 14 — Cross-cohort analyses: the article's figures

Figure 3 and Supplementary Figs. Sa–Sc for **one state set**, chosen in the next cell. No split by
rest and movie here: every person over all their scans, rest counted as one more task.

* **Figure 3.** Top: every cohort (occupancy, p(j | i), dynamic metrics). Bottom left: the
  reference transition graph (the cohorts' mean, each cohort once). Bottom right: how much each
  property depends on the task — ω² of task within cohort, every task of every cohort
  (< 0.01 stable, 0.01–0.06 moderate, ≥ 0.06 task-dependent), with P(tier) from the bootstrap.
* **Sa–Sc.** Task by task (rest included): dynamic metrics, occupancy, transitions.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from fmri_decomposition.unsupervised_model import cohort_tables as ct
from fmri_decomposition.figures import cohorts as cp

# ---- THE STATE SET: any atlas x window x clusterer stage 4b wrote (listed below)
ATLAS    = "harvardoxford"
WINDOW_S = 30.0
STATES   = "HMM2_umap3_8"          # e.g. "HMM2_pca3_8", "ThresholdCluster_umap3_8", "MeanShift_umap3_3"

# ---- panel C: data balance. "time" = every cohort the same HOURS of scan per draw (whole scans,
# pooled transition counts): people and stimulus duration balanced together. None = cohort means.
BALANCE  = "time"
HOURS    = 10
VERSUS   = "mean"                  # C: each cohort vs the mean of the others; "pairs" = every pair
N_BOOT   = 200                     # draws for panel C, bootstrap over people for panel D
PVALUE   = "stars"                 # Figure 3C confidence column: None, "exact" or "stars"; marker hollow when P(tier) < .95
TEST     = "tier"                  # "tier": P(tier), share of bootstrap |d| in the observed tier (* ≥.90, ** ≥.95, *** ≥.99);
                                    # "difference" (paired t), "tost" (equivalence ±0.2 SD), "both", "all"
COMBINE  = "random"                # "mixed": one mixed model, task share of variance; or ω² per cohort combined: "random", "fixed", "equal"
SAVE_DIR = None                    # e.g. "figures/14": PNG + SVG of every figure

cfg = ct.Settings(
    space=dict(atlas=ATLAS, window_s=WINDOW_S, states=STATES),
    entropy_estimator="miller_madow",
    distinct_minutes="min",   # distinct transitions rarefied to the shortest scan (None = as stored)
    cohorts={"ds002837": dict(dirs=["ds002837"], role="train"),
             "cneuromod": dict(dirs=["cneuromod"], role="train"),
             "camcan": dict(dirs=["camcan", "camcan_rest"], role="projected"),
             "hcp7t_fix": dict(dirs=["hcp7t_fix"], role="projected")},
    rest_pattern=r"rest\d*",          # Rest, REST1..4 (HCP); every other task is movie
    alpha=0.10,                        # BH-FDR (A.2, films)
    fc_task=1.0, fc_dyn=0.25,          # A.2: |log2FC| for the graph and the dynamics
    edge_top=12,                       # A.1/A.2: strongest arrows drawn
    sesoi_d=0.2,
    save_dir=SAVE_DIR,
)
sets = ct.available_sets(cfg.root, cfg.cohorts, cfg.modelling_version)
print(f"{len(sets)} state sets have every cohort; at {ATLAS}, {WINDOW_S:g} s:")
print(", ".join(sets[(sets.atlas == ATLAS) & (sets.window_s.astype(float) == WINDOW_S)]["states"]))
co = ct.Cohorts(cfg)
co.tasks()

## Figure 3

Every person over all their scans (all tasks, rest included). Top, every cohort; bottom left, the
reference graph (the cohorts' mean, each cohort once); bottom right, ω² of task within cohort.

In [ ]:
st = co.task_variation(n_boot=N_BOOT, combine=COMBINE)
vp = st if COMBINE == "mixed" else co.task_variation(n_boot=N_BOOT, combine="mixed")
vp_cells = co.task_variation(["transition_fit", *co.cells], n_boot=N_BOOT, combine="mixed")   # every p(j | i)
cp.fig_figure3(co, vp, st_transitions=vp_cells);   # bottom right: VPC task x VPC person


### Extra: Figure 3 with the task share and its tiers (previous bottom right)

In [ ]:
cp.fig_figure3(co, st, panel="tiers", pvalue=PVALUE, test=TEST, name="figure3_tiers");


### Variance partition (one figure on its own, and the table for the text)

From the mixed model: the share of each property's **total** variance due to cohort (variance of the cohort means, the fixed effect), task, person and residual. Beside each bar, the VPC of the task **within cohorts** (Figure 3, bottom right: cohort out of the denominator).

In [ ]:
cp.fig_variance_partition(co, vp, st_transitions=vp_cells);
cp.fig_variance_partition(co, vp, st_transitions=vp_cells, cohort=False, name="variance_partition_within");   # 3 colours


In [ ]:
vt = cp.variance_table(co, vp)
display(vt)
# print(vt.to_latex(index=False))   # for the manuscript


In [ ]:
# task vs person: x = VPC task, y = VPC person (within cohorts); small blue points = every transition
cp.fig_task_person(co, vp, st_transitions=vp_cells);


### Figure 3, alternative: one occupancy number

The same figure, with the occupancy of each state replaced in C by one per-person number: the correlation between their occupancy profile and the reference profile (as for the transition structure).

In [ ]:
cp.fig_figure3(co, st, panel="tiers", occupancy="fit", name="figure3_occupancy_fit", pvalue=PVALUE, test=TEST)
print(f"thinnest arrow drawn in B: p = {co.diagram_p_min:.2f}");


### Check: does one number per family lose information?

ω² of task for each family's profile (top row of each panel) and for every member: each transition (grouped by the state it leaves), each dynamic metric, each state's occupancy. Beside each row, the lowest and highest task mean.

In [ ]:
_, st_detail = cp.fig_aggregation_check(co, n_boot=N_BOOT, pvalue=PVALUE, test=TEST, combine=COMBINE)


In [ ]:
# why a transition row is missing above: people per cohort and condition with p(j | i) defined for each state
cov = co.row_coverage()
cov.pivot_table(index="state", columns=["cohort", "kind"], values=["with_row", "occupancy"]).round(3)


## Supplementary Figs. Sa–Sc

Sa–Sc: one row per task (rest, then every film, episode or run), grouped by cohort. Filled = the
same people did several tasks of their cohort (HCP, CNeuroMod, Cam-CAN); hollow = each film had its own
viewers (NNDb). Dashed blue = the global mean (every cohort and condition, each once). Occupancy
and transitions on 0–1; the dynamic metrics on the same fixed ranges as Figure 3.

In [ ]:
fs = co.film_summary(rest=True)
DYN = ["transition_fit", *co.dyn]
cp.fig_films_forest(co, fs, DYN, cp.global_means(co, DYN),
                    title="Supplementary Fig. Sa — dynamic metrics, task by task", name="Sa_films_dynamics");


In [ ]:
cp.fig_films_forest(co, fs, co.occ, cp.global_means(co, co.occ),
                    title="Supplementary Fig. Sb — occupancy of every state, task by task", name="Sb_films_occupancy",
                    labels={o: f"S{o[4:]} (share of time)" for o in co.occ});


In [ ]:
cp.fig_films_matrices(co, fs);


## Check: does a metric follow scan length?

Spearman ρ, within each cohort, between each metric and the minutes of scan behind it (one value per person and task). Entropy is corrected (Miller–Madow) and distinct transitions are rarefied to `co.distinct_minutes`; a metric still flagged here (|ρ| ≥ 0.3, FDR < 0.05) needs a correction too. Where films differ in length (NNDb), length and content go together: a flag, not a proof.

In [ ]:
print(f"distinct transitions rarefied to {co.distinct_minutes} min")
dc = co.duration_check()
flag = dc[(dc.rho.abs() >= .3) & (dc.p_fdr < .05)]
display(dc.pivot_table(index="variable", columns=["cohort", "kind"], values="rho").round(2))
flag.round(3)

## Numbers for the text

ω² of task per property (Figure 3, bottom right), and, task by task, how many task × property
comparisons differ from the other tasks of their cohort (Supplementary Sa–Sc, BH-FDR).

In [ ]:
display(st[["variable", "task_d", "task_lo", "task_hi", "tier", "task_p_tier", "task_mean_lo", "task_mean_hi", "n_tasks"]]
        .rename(columns={"task_d": "omega2", "task_lo": "lo", "task_hi": "hi", "task_p_tier": "P_tier"}).round(3))
fs.groupby(fs["variable"].map(lambda v: "transition" if "->" in v or v == "transition_fit" else
                              "occupancy" if v.startswith("occ_") else "dynamics"))["differs"].agg(["sum", "count"])
